In [ ]:
library(tidyverse)
library(bigrquery)

drug_01846335_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20251015",  # Comment out this line if you want the export to always overwrite.
  "drug_01846335",
  "drug_01846335_*.csv")
message(str_glue('The data will be written to {drug_01846335_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), drug_type_concept_name = col_character(), stop_reason = col_character(), sig = col_character(), route_concept_name = col_character(), lot_number = col_character(), visit_occurrence_concept_name = col_character(), drug_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), route_source_value = col_character(), dose_unit_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_01846335_drug_df <- read_bq_export_from_workspace_bucket(drug_01846335_path)

In [ ]:
## Added in oral contraceptives
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_53214561_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20251204", 
  "drug_53214561",
  "drug_53214561_*.csv")
message(str_glue('The data will be written to {drug_53214561_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {drug_53214561_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), drug_type_concept_name = col_character(), stop_reason = col_character(), sig = col_character(), route_concept_name = col_character(), lot_number = col_character(), visit_occurrence_concept_name = col_character(), drug_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), route_source_value = col_character(), dose_unit_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_53214561_drug_df <- read_bq_export_from_workspace_bucket(drug_53214561_path)

In [ ]:
drug_44902289_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260216",  # Comment out this line if you want the export to always overwrite.
  "drug_44902289",
  "drug_44902289_*.csv")
message(str_glue('The data will be written to {drug_44902289_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {drug_44902289_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), drug_type_concept_name = col_character(), stop_reason = col_character(), sig = col_character(), route_concept_name = col_character(), lot_number = col_character(), visit_occurrence_concept_name = col_character(), drug_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), route_source_value = col_character(), dose_unit_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
neg <- read_bq_export_from_workspace_bucket(drug_44902289_path)

In [ ]:
# Load required libraries
library(dplyr)
library(stringr)

medication_df <- dataset_53214561_drug_df

In [ ]:
#--- Extract (none oral contraceptive) inhibitors/inducers
known_drugs <- c("wort","efavirenz", "enzalutamide", "letermovir", "(prednisone", "prednisone", "rifampin", "(ritonavir", "ritonavir",
                "armodafinil", "chloramphenicol", "cimetidine", "esomeprazole", "felbamate", "fluconazole", 
                "fluoxetine", "fluvoxamine", "isoniazid", "(ketoconazole", "ketoconazole", "luliconazole", "modafinil", "(omeprazole", "omeprazole",
                "oritavancin", "quercetin", "rucaparib", "ticlopidine", "topiramate", "voriconazole",
                "artemisinin", "carbamazepine", "dabrafenib", "efavirenz", "nevirapine", "phenobarbital",
                "phenytoin", "rifampin", "clopidogrel", "thiotepa", "ticlopidine", "voriconazole",
                "abiraterone", "amiodarone", "bupropion", "celecoxib", "chlorpromazine", "cinacalcet", "citalopram",
                "clemastine", "clobazam", "clomipramine", "cocaine", "diphenhydramine", "(diphenhydramine", "doxepin", "duloxetine", 
                 "escitalopram", "fluoxetine", "haloperidol", "hydroxychloroquine", "hydroxyzine",
                 "lorcaserin", "methadone", "metoclopramide", "midodrine", "panobinostat", "paroxetine",
                 "perphenazine", "promethazine", "quinidine", "ritonavir", "rolapitant", "sertraline", "terbinafine", "(terbinafine", 
                 "ticlopidine", "tripelennamine", "vemurafenib", "cenobamate", "halofantrine", "levomepromazine", "moclobemide", "perhexiline")

#-- extract oral contraceptives
#-- progesterones and estrogens, fixed combinations
g03aa_lookup <- data.frame(
  code = c("22656", "22968", "11636", "2539031", "4083", "4124", "24591", "6373", 
           "6691", "6703", "326374", "7514", "31994", "7518"),
  drug_name = c("desogestrel", "dienogest", "drospirenone", "estetrol", "estradiol", "ethinyl estradiol", 
               "ethynodiol", "levonorgestrel", "medroxyprogesterone", "megestrol", "norelgestromin", "norethindrone",
               "norgestimate", "norgestrel")
)
#-- progesterones and estrogens, sequential preparations
g03ab_lookup <- data.frame(
     code = c("22656", "22968", "4083", "4124", "6373", "6703", "7514", "31994"),
    drug_name = c("desogestrel", "dienogest", "estradiol", "ethinyl estradiol", "levonorgestrel",
                 "megestrol", "norethindrone", "norgestimate")
    )
#-- progesterones
g03ac_lookup <- data.frame(
    code = c("22656", "11636", "14584", "6373", "6691", "6703", "7514"),
    drug_name = c("desogestrel", "drospirenone", "etonogestrel", "levonorgestrel", "medroxyprogesterone",
                 "megestrol", "norethindrone")
    )
#-- emergency contraceptives
g03ad_lookup <- data.frame(
    code = c("6373", "1005921"),
    drug_name = c("levonorgestrel", "ulipristal")
    )

# Combine all lookup dataframes
contraceptive_lookup <- rbind(g03aa_lookup, g03ab_lookup, g03ac_lookup, g03ad_lookup)

# Remove duplicate rows
contraceptive_lookup <- unique(contraceptive_lookup)


In [ ]:
# Function to extract drug name from any position in the split string
extract_name <- function(concept_name) {
  # Split the string and convert to lowercase
  words <- tolower(strsplit(concept_name, " ", fixed = TRUE)[[1]])
  
  # Find which word matches our known drugs
  match_idx <- which(words %in% known_drugs)
  
  # Return the first match, or NA if no match found
  if (length(match_idx) > 0) {
    return(words[match_idx[1]])
  } else {
    return(NA_character_)
  }
}

#-- Apply the function to extract drug names
medication_df$drug <- sapply(medication_df$standard_concept_name, extract_name)

#-- Filter out rows where no cyp modulator was found (NA values)
cyp_df <- medication_df[!is.na(medication_df$drug), ]

In [ ]:
#-- Clean antidepressant names
cyp_df <- cyp_df %>%
  mutate(
    drug = case_when(
      drug == "bupropion" ~ "NDRI:bupropion",
      drug == "clomipramine" ~ "TCA:clomipramine",
      drug == "doxepin" ~ "TCA:doxepin",
      drug == "citalopram" ~ "SSRI:citalopram",
      drug == "duloxetine" ~ "SNRI:duloxetine",
      drug == "escitalopram" ~ "SSRI:escitalopram",
      drug == "fluoxetine" ~ "SSRI:fluoxetine",
      drug == "fluvoxamine" ~ "SSRI:fluvoxamine",
      drug == "paroxetine" ~ "SSRI:paroxetine",
      drug == "sertaline" ~ "SSRI:sertraline",
      drug == "(prednisone" ~ "prednisone",
      drug == "(ritonavir" ~ "ritonavir",
      drug == "(ketoconazole" ~ "ketoconazole",
      drug == "(omeprazole" ~ "omeprazole",
      drug == "wort" ~ "St. Johns wort extract",
      drug == "(diphenhydramine" ~ "diphenhydramine",
      drug == "(terbinafine" ~ "terbinafine",
      drug == "moclobemide" ~ "RIMA:moclobemide",
      TRUE ~ drug
    )
  )

#--- Format CYP modulator scripts
cyp_scripts <- cyp_df %>%
  select(person_id, drug, drug_exposure_start_datetime, drug_exposure_end_datetime, days_supply, refills)

In [ ]:
#-- Extract oral contraceptive scripts and combine with CYP scripts
oral_con <- medication_df %>%
    filter(standard_concept_code %in% contraceptive_lookup$code) %>%
    select(person_id, standard_concept_code, drug_exposure_start_datetime, drug_exposure_end_datetime, days_supply, refills)

#-- add drug names with left join
oral_con <- oral_con %>%
  left_join(contraceptive_lookup, by = c("standard_concept_code" = "code")) %>%
  select(person_id, drug_name, drug_exposure_start_datetime, drug_exposure_end_datetime, days_supply, refills) %>%
  rename(drug = drug_name)

#-- All modulator scripts
mod_scripts <- rbind(cyp_scripts, oral_con)

In [ ]:
#--repeat for negative controls
known_drugs <- c("gabapentin", "levothyroxine", "pravastatin", "rosuvastatin")

# Function to extract drug name from any position in the split string
extract_name <- function(concept_name) {
  # Split the string and convert to lowercase
  words <- tolower(strsplit(concept_name, " ", fixed = TRUE)[[1]])
  
  # Find which word matches our known drugs
  match_idx <- which(words %in% known_drugs)
  
  # Return the first match, or NA if no match found
  if (length(match_idx) > 0) {
    return(words[match_idx[1]])
  } else {
    return(NA_character_)
  }
}

#-- Apply the function to extract drug names
neg$drug <- sapply(neg$standard_concept_name, extract_name)

#-- Filter out rows where no cyp modulator was found (NA values)
neg_df <- neg[!is.na(neg$drug), ]

In [ ]:
#--- Format CYP modulator scripts
neg_scripts <- neg_df %>%
  select(person_id, drug, drug_exposure_start_datetime, drug_exposure_end_datetime, days_supply, refills)


In [ ]:
#=== Categorise (including antidepressants)
cyp2c19_inducers_drugs <- c("St. Johns wort extract", "efavirenz", "enzalutamide", "letermovir", 
                      "prednisone", "rifampin", "ritonavir")

cyp2c19_inhibitors_drugs <- c("armodafinil", "chloramphenicol", "cimetidine", "esomeprazole", "felbamate", "fluconazole", 
                "isoniazid", "ketoconazole", "luliconazole", "modafinil", "omeprazole", "SSRI:fluoxetine", "SSRI:fluvoxamine",
                "oritavancin", "quercetin", "rucaparib", "ticlopidine", "topiramate", "voriconazole", "cenobamate") # has antidepressants

cyp2b6_inducers_drugs <- c("artemisinin", "carbamazepine", "dabrafenib", "efavirenz", "nevirapine", "phenobarbital",
                "phenytoin", "rifampin")

cyp2b6_inhibitors_drugs <- c("clopidogrel", "thiotepa", "ticlopidine", "voriconazole")

cyp2d6_inhibitors_drugs <- c("abiraterone", "amiodarone", "celecoxib", "chlorpromazine", "cinacalcet",
                "clemastine", "clobazam", "cocaine", "diphenhydramine", "cimetidine",
                 "haloperidol", "hydroxychloroquine", "hydroxyzine",
                 "lorcaserin", "methadone", "metoclopramide", "midodrine", "panobinostat", 
                 "perphenazine", "promethazine", "quinidine", "ritonavir", "rolapitant", "terbinafine", 
                 "ticlopidine", "tripelennamine", "vemurafenib", "NDRI:bupropion", "SSRI:citalopram", "TCA:clomipramine",
                            "TCA:doxepin", "SNRI:duloxetine", "SSRI:escitalopram", "SSRI:fluoxetine", "SSRI:sertraline",
                            "SSRI:paroxetine", "RIMA:moclobemide", "halofantrine", "levomepromazine", "perhexiline") #has antidepressants

#== Inhibitors/inducers no matter strength of modulation
cyp2c19_inducers <- mod_scripts %>%
    filter(drug %in% cyp2c19_inducers_drugs)
cyp2c19_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2c19_inhibitors_drugs)
cyp2b6_inducers <- mod_scripts %>%
    filter(drug %in% cyp2b6_inducers_drugs)
cyp2b6_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2b6_inhibitors_drugs)
cyp2d6_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2d6_inhibitors_drugs)

#== Inhibitors/inducers (no AD)
cyp2c19_inhibitors_noAD <- cyp2c19_inhibitors %>% filter(!drug %in% c("SSRI:fluoxetine", "SSRI:fluvoxamine"))
cyp2d6_inhibitors_noAD <- cyp2d6_inhibitors %>% filter(!drug %in% c("NDRI:bupropion", "SSRI:citalopram", "TCA:clomipramine",
                            "TCA:doxepin", "SNRI:duloxetine", "SSRI:escitalopram", "SSRI:fluoxetine", "SSRI:sertraline",
                            "SSRI:paroxetine", "RIMA:moclobemide"))

#== AD inhibitors/inducers
cyp2c19_inhibitors_AD <- cyp2c19_inhibitors %>% filter(drug %in% c("SSRI:fluoxetine", "SSRI:fluvoxamine"))
cyp2d6_inhibitors_AD <- cyp2d6_inhibitors %>% filter(drug %in% c("NDRI:bupropion", "SSRI:citalopram", "TCA:clomipramine",
                            "TCA:doxepin", "SNRI:duloxetine", "SSRI:escitalopram", "SSRI:fluoxetine", "SSRI:sertraline",
                            "SSRI:paroxetine", "RIMA:moclobemide"))

In [ ]:
#== Categorise by strength of modulator
#-- Put those with in-vitro evidence only and TBD inhibitor strength level under review into the weak categories
cyp2c19_strong_inhibitors_drugs <- c("fluconazole", "SSRI:fluvoxamine", "ticlopidine") # AD
cyp2c19_moderate_inhibitors_drugs <- c("cenobamate", "esomeprazole", "SSRI:fluoxetine", "ketoconazole", "voriconazole") #AD
cyp2c19_weak_inhibitors_drugs <- c(c("armodafinil", "chloramphenicol", "cimetidine", "felbamate", "isoniazid", "luliconazole",
                            "modafinil", "omeprazole", "rucaparib", "oritavancin", "quercetin", "topiramate"), contraceptive_lookup$drug_name) # + oral contraceptives

cyp2b6_strong_inhibitors_drugs <- c("ticlopidine")
cyp2b6_moderate_inhibitors_drugs <- c("voriconazole")
cyp2b6_weak_inhibitors_drugs <- c("clopidogrel", "thiotepa")

cyp2d6_strong_inhibitors_drugs <- c("NDRI:bupropion", "SSRI:fluoxetine", "SSRI:paroxetine", "perhexiline", "quinidine") #AD
cyp2d6_moderate_inhibitors_drugs <- c("abiraterone", "cinacalcet", "clobazam", "TCA:doxepin", "SNRI:duloxetine", 
                               "halofantrine", "lorcaserin", "RIMA:moclobemide", "rolapitant", "terbinafine") # AD
cyp2d6_weak_inhibitors_drugs <- c("amiodarone", "celecoxib", "cimetidine", "chlorpromazine", "SSRI:citalopram", "clemastine",
                           "TCA:clomipramine", "diphenhydramine", "SSRI:escitalopram", "hydroxychloroquine", "vemurafenib",
                           "levomepromazine", "cocaine", "haloperidol", "hydroxyzine", "methadone", "metoclopramide",
                           "midodrine", "panobinostat", "perphenazine", "promethazine", "ticlopidine", "tripelennamine", "ritonavir", "SSRI:sertraline")
# AD
cyp2b6_moderate_inducers_drugs <- c("artemisinin", "carbamazepine", "efavirenz", "rifampin")
cyp2b6_weak_inducers_drugs <- c("nevirapine", "dabrafenib", "phenobarbital", "phenytoin")

cyp2c19_moderate_inducers_drugs <- c("efavirenz", "enzalutamide", "rifampin", "St. Johns wort extract")
cyp2c19_weak_inducers_drugs <- c("letermovir", "prednisone", "ritonavir")

#== Inhibitors/inducers with strength accounted for
cyp2c19_strong_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2c19_strong_inhibitors_drugs)
cyp2c19_moderate_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2c19_moderate_inhibitors_drugs)
cyp2c19_weak_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2c19_weak_inhibitors_drugs)

cyp2b6_strong_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2b6_strong_inhibitors_drugs)
cyp2b6_moderate_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2b6_moderate_inhibitors_drugs)
cyp2b6_weak_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2b6_weak_inhibitors_drugs)

cyp2d6_strong_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2d6_strong_inhibitors_drugs)
cyp2d6_moderate_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2d6_moderate_inhibitors_drugs)
cyp2d6_weak_inhibitors <- mod_scripts %>%
    filter(drug %in% cyp2d6_weak_inhibitors_drugs)

cyp2b6_moderate_inducers <- mod_scripts %>%
    filter(drug %in% cyp2b6_moderate_inducers_drugs)
cyp2b6_weak_inducers <- mod_scripts %>%
    filter(drug %in% cyp2b6_weak_inducers_drugs)

cyp2c19_moderate_inducers <- mod_scripts %>%
    filter(drug %in% cyp2c19_moderate_inducers_drugs)
cyp2c19_weak_inducers <- mod_scripts %>%
    filter(drug %in% cyp2c19_weak_inducers_drugs)

# Inhibitors/inducers that are not ADs
cyp2c19_strong_inhibitors_noAD <- cyp2c19_strong_inhibitors %>%
    filter(!drug %in% c("SSRI:fluvoxamine"))
cyp2c19_moderate_inhibitors_noAD <- cyp2c19_moderate_inhibitors %>%
    filter(!drug %in% c("SSRI:fluoxetine"))
cyp2d6_strong_inhibitors_noAD <- cyp2d6_strong_inhibitors %>%
    filter(!drug %in% c("SSRI:fluoxetine", "SSRI:paroxetine"))
cyp2d6_moderate_inhibitors_noAD <- cyp2d6_moderate_inhibitors %>%
    filter(!drug %in% c("TCA:doxepin", "SNRI:duloxetine", "RIMA:moclobemide"))
cyp2d6_weak_inhibitors_noAD <- cyp2d6_weak_inhibitors %>%
    filter(!drug %in% c("SSRI:citalopram", "TCA:clomipramine", "SSRI:escitalopram", "SSRI:sertraline"))

# Inhibitors/inducers that are ADs
cyp2c19_strong_inhibitors_AD <- cyp2c19_strong_inhibitors %>%
    filter(drug %in% c("SSRI:fluvoxamine"))
cyp2c19_moderate_inhibitors_AD <- cyp2c19_moderate_inhibitors %>%
    filter(drug %in% c("SSRI:fluoxetine"))
cyp2d6_strong_inhibitors_AD <- cyp2d6_strong_inhibitors %>%
    filter(drug %in% c("SSRI:fluoxetine", "SSRI:paroxetine", "NDRI:bupropion"))
cyp2d6_moderate_inhibitors_AD <- cyp2d6_moderate_inhibitors %>%
    filter(drug %in% c("TCA:doxepin", "SNRI:duloxetine", "RIMA:moclobemide"))
cyp2d6_weak_inhibitors_AD <- cyp2d6_weak_inhibitors %>%
    filter(drug %in% c("SSRI:citalopram", "TCA:clomipramine", "SSRI:escitalopram", "SSRI:sertraline"))

In [ ]:
#-- Categorise the Negative controls by each drug
rosuvastatin_df <- neg_scripts %>%
    filter(drug == "rosuvastatin")
levothyroxine_df <- neg_scripts %>%
    filter(drug == "levothyroxine")
pravastatin_df <- neg_scripts %>%
    filter(drug == "pravastatin")
gabapentin_df <- neg_scripts %>%
    filter(drug == "gabapentin")

In [ ]:
#-- Load outcomes
Outcome_raw <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_2scripts_S_C.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence),
    DateFirstDispense = as.Date(DateFirstDispense),
    Index = as.Date(Index),
    cyp_concomitant_upper_window = as.Date(cyp_concomitant_upper_window)
  )


In [ ]:
library(data.table)

## Convert to data.table
setDT(Outcome_raw)

## If your modulator tables are data.frames:
setDT(cyp2c19_inducers)
setDT(cyp2c19_inhibitors)
setDT(cyp2c19_inhibitors_noAD)
setDT(cyp2c19_inhibitors_AD)
setDT(cyp2b6_inducers)
setDT(cyp2b6_inhibitors)
setDT(cyp2d6_inhibitors)
setDT(cyp2d6_inhibitors_noAD)
setDT(cyp2d6_inhibitors_AD)
setDT(cyp2c19_strong_inhibitors)
setDT(cyp2c19_moderate_inhibitors)
setDT(cyp2c19_weak_inhibitors)
setDT(cyp2b6_strong_inhibitors)
setDT(cyp2b6_moderate_inhibitors)
setDT(cyp2b6_weak_inhibitors)
setDT(cyp2d6_strong_inhibitors)
setDT(cyp2d6_moderate_inhibitors)
setDT(cyp2d6_weak_inhibitors)
setDT(cyp2b6_moderate_inducers)
setDT(cyp2b6_weak_inducers)
setDT(cyp2c19_moderate_inducers)
setDT(cyp2c19_weak_inducers)
setDT(cyp2c19_strong_inhibitors_noAD)
setDT(cyp2c19_moderate_inhibitors_noAD)
setDT(cyp2d6_strong_inhibitors_noAD)
setDT(cyp2d6_moderate_inhibitors_noAD)
setDT(cyp2d6_weak_inhibitors_noAD)
setDT(cyp2c19_strong_inhibitors_AD)
setDT(cyp2c19_moderate_inhibitors_AD)
setDT(cyp2d6_strong_inhibitors_AD)
setDT(cyp2d6_moderate_inhibitors_AD)
setDT(cyp2d6_weak_inhibitors_AD)

setDT(rosuvastatin_df)
setDT(levothyroxine_df)
setDT(pravastatin_df)
setDT(gabapentin_df)

In [ ]:
baseline_days <- 60L

dat <- Outcome_raw[
  , .(
      person_id,
      drug.treatment,
      Index = as.IDate(Index),
      cyp_concomitant_upper_window = as.IDate(cyp_concomitant_upper_window)
    )
]
setkey(dat, person_id)


In [ ]:
mod_tables <- list(
  cyp2c19_inducers                = cyp2c19_inducers,
  cyp2c19_inhibitors              = cyp2c19_inhibitors,
  cyp2c19_inhibitors_noAD         = cyp2c19_inhibitors_noAD,
  cyp2c19_inhibitors_AD           = cyp2c19_inhibitors_AD,
  cyp2b6_inducers                 = cyp2b6_inducers,
  cyp2b6_inhibitors               = cyp2b6_inhibitors,
  cyp2d6_inhibitors               = cyp2d6_inhibitors,
  cyp2d6_inhibitors_noAD          = cyp2d6_inhibitors_noAD,
  cyp2d6_inhibitors_AD            = cyp2d6_inhibitors_AD,
  cyp2c19_strong_inhibitors       = cyp2c19_strong_inhibitors,
  cyp2c19_moderate_inhibitors     = cyp2c19_moderate_inhibitors,
  cyp2c19_weak_inhibitors         = cyp2c19_weak_inhibitors,
  cyp2b6_strong_inhibitors        = cyp2b6_strong_inhibitors,
  cyp2b6_moderate_inhibitors      = cyp2b6_moderate_inhibitors,
  cyp2b6_weak_inhibitors          = cyp2b6_weak_inhibitors,
  cyp2d6_strong_inhibitors        = cyp2d6_strong_inhibitors,
  cyp2d6_moderate_inhibitors      = cyp2d6_moderate_inhibitors,
  cyp2d6_weak_inhibitors          = cyp2d6_weak_inhibitors,
  cyp2b6_moderate_inducers        = cyp2b6_moderate_inducers,
  cyp2b6_weak_inducers            = cyp2b6_weak_inducers,
  cyp2c19_moderate_inducers       = cyp2c19_moderate_inducers,
  cyp2c19_weak_inducers           = cyp2c19_weak_inducers,
  cyp2c19_strong_inhibitors_noAD  = cyp2c19_strong_inhibitors_noAD,
  cyp2c19_moderate_inhibitors_noAD= cyp2c19_moderate_inhibitors_noAD,
  cyp2d6_strong_inhibitors_noAD   = cyp2d6_strong_inhibitors_noAD,
  cyp2d6_moderate_inhibitors_noAD = cyp2d6_moderate_inhibitors_noAD,
  cyp2d6_weak_inhibitors_noAD     = cyp2d6_weak_inhibitors_noAD,
  cyp2c19_strong_inhibitors_AD    = cyp2c19_strong_inhibitors_AD,
  cyp2c19_moderate_inhibitors_AD  = cyp2c19_moderate_inhibitors_AD,
  cyp2d6_strong_inhibitors_AD     = cyp2d6_strong_inhibitors_AD,
  cyp2d6_moderate_inhibitors_AD   = cyp2d6_moderate_inhibitors_AD,
  cyp2d6_weak_inhibitors_AD       = cyp2d6_weak_inhibitors_AD,
  rosuvastatin_neg_cont           = rosuvastatin_df,
  levothyroxine_neg_cont          = levothyroxine_df,
  pravastatin_neg_cont            = pravastatin_df,
  gabapentin_neg_cont             = gabapentin_df
)

mod_lookup <- rbindlist(
  mod_tables,
  idcol = "mod_name",
  use.names = TRUE,
  fill = TRUE
)[
  , .(
      person_id,
      drug,  # modulator drug
      drug_exposure_start_datetime = as.IDate(drug_exposure_start_datetime),
      mod_name
    )
]

setkey(mod_lookup, person_id)


In [ ]:
# join dat (index etc.) with all modulators for that person
mod_flags_long <- mod_lookup[
  dat,
  on = "person_id",
  allow.cartesian = TRUE  # many-to-many
][
  # define windows
  , `:=`(
      in_baseline = (
        drug_exposure_start_datetime <= Index &
        drug_exposure_start_datetime >  Index - baseline_days
      ),
      in_concom = (
        drug_exposure_start_datetime >  Index &
        drug_exposure_start_datetime < cyp_concomitant_upper_window
      )
    )
][
  # keep only exposures in either window & not the same drug
  (in_baseline | in_concom) & (drug.treatment != drug)
][
  # now aggregate per person + modulator
  , .(
      baseline_any = any(in_baseline),
      concom_any   = any(in_concom)
    ),
    by = .(person_id, mod_name)
][
  # derive your three flags
  , `:=`(
      any_followup  = baseline_any | concom_any,
      baseline_only = baseline_any & !concom_any,
      concom_only   = concom_any & !baseline_any
    )
][
  # keep only the three final indicators
  , .(person_id, mod_name, any_followup, baseline_only, concom_only)
]


In [ ]:
# convert logical -> integer before casting, so fill=0 works nicely
mod_flags_long[
  , c("any_followup", "baseline_only", "concom_only") :=
      lapply(.SD, as.integer),
  .SDcols = c("any_followup", "baseline_only", "concom_only")
]

mod_flags_wide <- dcast(
  mod_flags_long,
  person_id ~ mod_name,
  value.var = c("any_followup", "baseline_only", "concom_only"),
  fill = 0
)


In [ ]:
setkey(mod_flags_wide, person_id)
setkey(Outcome_raw, person_id)
Outcome_raw_flagged <- mod_flags_wide[Outcome_raw]

# Check BEFORE
message("NAs BEFORE fix:")
Outcome_raw_flagged[, lapply(.SD, function(x) sum(is.na(x))), .SDcols = patterns("^(any_followup|baseline_only|concom_only)_")] |> print()

# Fix
mod_cols <- grep("^(any_followup|baseline_only|concom_only)_", names(Outcome_raw_flagged), value = TRUE)
Outcome_raw_flagged[, (mod_cols) := lapply(.SD, function(x) replace(x, is.na(x), 0L)), .SDcols = mod_cols]

# Check AFTER
message("NAs AFTER fix:")
Outcome_raw_flagged[, lapply(.SD, function(x) sum(is.na(x))), .SDcols = patterns("^(any_followup|baseline_only|concom_only)_")] |> print()

In [ ]:
#-- Save results for first analyses. Participants can have more than one classification.
write.csv(Outcome_raw_flagged, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_CypModulators_2scripts_S_C.csv", row.names = FALSE, quote = FALSE)

# Copy to workspace bucket
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp /home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_CypModulators_2scripts_S_C.csv",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/Main_CYP_Outcomes_90days_CypModulators_2scripts_S_C.csv")
))


In [ ]:
#-- sanity check
Outcome_raw_flagged %>%
  group_by(person_id, drug.treatment) %>%
  filter(n() > 1) %>%
  nrow()
